In [27]:
import pandas as pd
import numpy as np
from scipy.stats import norm

# 1. Paramètres de l'énoncé
R = 0.40             # Recovery
zc_xyz = 0.0200      # Spread XYZ (200 bps)
zc_bank = 0.0070     # Spread Banque (70 bps)
nominal = 10_000_000 # 10M €
dv01_total = 14000   # DV01 donnée
n_years = 20         # Maturité 20Y
vol_swaption = 0.25  # Volatilité 25%
strike = 0.0394      # Taux ATM (Swap 20Y)

# 2. Chargement et nettoyage du CSV
df_zc = pd.read_csv('Data(ZC).csv', sep=';')
def clean_val(col):
    return pd.to_numeric(col.astype(str).str.replace(',', '.').str.replace('%', ''), errors='coerce')

df_zc['Ti'] = clean_val(df_zc['Unnamed: 1'])
df_zc['B0_Ti'] = clean_val(df_zc['B(0,T)'])

print("Initialisation terminée.")

Initialisation terminée.


In [30]:
df_zc

,Unnamed: 0,Unnamed: 1,Cot SHIFT,ZC SHIFT,"B(0,T)",Ti,B0_Ti
0,MM,"0,25","3,0698%","3,0581%","0,992384019",0.25,0.992384
1,MM,"0,50","2,6191%","2,6021%","0,987073719",0.50,0.987074
2,MM,"0,75","2,3958%","2,3745%","0,982348717",0.75,0.982349
3,MM,"1,00","2,2979%","2,2719%","0,977537139",1.00,0.977537
4,FUT,"1,25","97,8691%","2,2426%","0,972357242",1.25,0.972357
5,FUT,"1,50","97,7094%","2,2495%","0,966820787",1.50,0.966821
6,FUT,"1,75","97,4981%","2,2844%","0,960811246",1.75,0.960811
7,FUT,"2,00","97,2911%","2,3363%","0,95434817",2.00,0.954348
8,FUT,"2,25","97,0984%","2,3980%","0,947475179",2.25,0.947475
9,FUT,"2,50","96,9711%","2,4599%","0,94035453",2.50,0.940355


## Question 1 ##

In [19]:
# Création de l'échéancier 0 à 20
maturities = np.arange(0, n_years + 1, 1)
df_q = pd.DataFrame({'Ti': maturities})

# Question 1 : XYZ
hr_xyz = zc_xyz / (1 - R)
df_q['SP_XYZ'] = np.exp(-hr_xyz * df_q['Ti'])
df_q['DP_XYZ'] = df_q['SP_XYZ'].shift(1) - df_q['SP_XYZ']
df_q.loc[0, 'DP_XYZ'] = 0

# Question 2 : Banque
hr_bank = zc_bank / (1 - R)
df_q['SP_Bank'] = np.exp(-hr_bank * df_q['Ti'])

print("Questions 1 & 2 : OK")
print(df_q[['Ti', 'SP_XYZ', 'DP_XYZ', 'SP_Bank']].head(30))

Questions 1 & 2 : OK
    Ti    SP_XYZ    DP_XYZ   SP_Bank
0    0  1.000000  0.000000  1.000000
1    1  0.967216  0.032784  0.988401
2    2  0.935507  0.031709  0.976937
3    3  0.904837  0.030670  0.965605
4    4  0.875173  0.029664  0.954405
5    5  0.846482  0.028692  0.943335
6    6  0.818731  0.027751  0.932394
7    7  0.791890  0.026841  0.921579
8    8  0.765928  0.025961  0.910890
9    9  0.740818  0.025110  0.900325
10  10  0.716531  0.024287  0.889882
11  11  0.693041  0.023491  0.879560
12  12  0.670320  0.022721  0.869358
13  13  0.648344  0.021976  0.859275
14  14  0.627089  0.021255  0.849308
15  15  0.606531  0.020558  0.839457
16  16  0.586646  0.019884  0.829720
17  17  0.567414  0.019233  0.820096
18  18  0.548812  0.018602  0.810584
19  19  0.530819  0.017992  0.801182
20  20  0.513417  0.017402  0.791890


SP_XYZ (Survival Probability) : C'est la probabilité que la contrepartie XYZ n'ait pas fait défaut entre $T_0$ et la date $T_i$. On remarque qu'à $T=0$, elle est de 100% (certitude de survie au départ) et qu'elle décroît de manière exponentielle avec le temps.


DP_XYZ (Default Probability) : C'est la probabilité de défaut marginale. Elle représente la probabilité que XYZ fasse défaut spécifiquement au cours de l'année $i$, sachant qu'elle était en vie au début de cette année.

## Question 2 ##

In [ ]:
# Paramètre de crédit de la Banque
zc_bank = 0.0070  # Spread de crédit de la banque (70 bps) 

# Calcul de la probabilité de survie de la Banque (Survival Probability)
# Formule : SP = exp( - (zc_bank / (1-R)) * Ti ) 
hazard_rate_bank = zc_bank / (1 - R)
df_q['SP_Bank'] = np.exp(-hazard_rate_bank * df_q['Ti'])

print("Résultats : Probabilités de survie de la Banque")
print(df_q[['Ti', 'SP_Bank']].head(10))

Résultats : Probabilités de survie de la Banque
   Ti   SP_Bank
0   0  1.000000
1   1  0.988401
2   2  0.976937
3   3  0.965605
4   4  0.954405
5   5  0.943335
6   6  0.932394
7   7  0.921579
8   8  0.910890
9   9  0.900325


## Question 3

In [21]:
def black_formula(F, K, sigma, T):
    if T <= 0: return 0
    d1 = (np.log(F / K) + 0.5 * sigma**2 * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    return F * norm.cdf(d1) - K * norm.cdf(d2)

epe_list = []
for t in maturities:
    if t == 0:
        epe_list.append(0)
    else:
        dv01_res = dv01_total * (n_years - t) / n_years
        # Black donne un prix en %, on multiplie par 10000 pour avoir des bps, puis par la DV01
        price_black = black_formula(strike, strike, vol_swaption, t)
        epe_list.append(price_black * 10000 * dv01_res)

df_q['EPE'] = epe_list
print("Question 3 : Profil EPE calculé.")
df_q[['Ti', 'EPE']].head(30)

Question 3 : Profil EPE calculé.


,Ti,EPE
0,0,0.000000
1,1,521276.491510
2,2,696585.767116
3,3,803658.507530
4,4,871142.547947
5,5,910741.445479
6,6,928761.984195
7,7,929131.978031
8,8,914529.820901
9,9,886899.676263


**Interprétation du profil de l'EPE (Expected Positive Exposure)**
Le profil que vous avez obtenu est une courbe dite en "cloche" (ou hump-shaped), typique des produits de taux comme les swaps. Plusieurs phénomènes financiers expliquent cette forme :

**Le point de départ (Ti = 0)** : L'EPE est nulle car le swap est conclu "At-the-money" (à la monnaie). Sa valeur de marché initiale est donc nulle, et il n'y a pas encore d'incertitude sur l'évolution des taux.

**La phase de croissance (Années 1 à 7)** : L'EPE augmente car l'incertitude sur les taux d'intérêt s'accroît avec le temps. C'est l'effet de diffusion : plus on s'éloigne du présent, plus la probabilité que les taux s'écartent du taux fixe initial (le Strike) est grande, ce qui augmente l'exposition potentielle de la banque.

**Le pic d'exposition (Année 7) :** À environ 929 132 €, le risque est à son maximum. C'est le point d'équilibre où l'incertitude est forte, mais où il reste encore une durée de vie résiduelle importante (13 ans).

**La phase de décroissance (Années 8 à 20) :** L'exposition diminue progressivement. C'est l'effet d'amortissement (amortization effect). Même si les taux deviennent très volatils, il reste de moins en moins de flux financiers à échanger jusqu'à la maturité. La sensibilité du swap (sa DV01 résiduelle) chute, ce qui réduit mécaniquement l'exposition.

**Le point final (Ti = 20) :** À l'échéance, l'exposition redevient nulle car tous les flux ont été échangés.

## Question 4

In [32]:
# Extraction des B(0,Ti) réels
actualisation_factors = []


for m in maturities:
    # 1. Cas particulier de l'année 0 (Absente de l'Excel)
    if m == 0:
        actualisation_factors.append(1.0) # 1€ aujourd'hui vaut 1€
        continue

    # 2. Recherche de l'année m dans ton Excel
    # On utilise .round(0) pour être sûr de matcher "1", "2", etc.
    matching_row = df_zc[df_zc['Ti'].round(0) == m]
    
    if not matching_row.empty:
        valeur = matching_row['B0_Ti'].values[0]
        actualisation_factors.append(valeur)
    else:
        # 3. Sécurité au cas où une année manque (ex: l'année 15 est absente)
        idx_proche = (df_zc['Ti'] - m).abs().idxmin()
        valeur = df_zc.loc[idx_proche, 'B0_Ti']
        actualisation_factors.append(valeur)

df_q['B0_Ti_Real'] = actualisation_factors

# Formule de la CVA
df_q['CVA_component'] = (1 - R) * df_q['B0_Ti_Real'] * df_q['EPE'] * df_q['DP_XYZ'] * df_q['SP_Bank']
cva_total = df_q['CVA_component'].sum()

print(f"--- RÉSULTATS QUESTION 4 ---")
print(f"CVA Totale : {cva_total:,.2f} €")
print(f"CVA Upfront (bps) : {(cva_total/nominal)*10000:.2f} bps")

--- RÉSULTATS QUESTION 4 ---
CVA Totale : 140,068.43 €
CVA Upfront (bps) : 140.07 bps


## Question 5


In [33]:
# --- QUESTION 5 : CALCUL DES SENSIBILITÉS ---

def compute_cva_sensi(vol_adj=0, spread_xyz_adj=0):
    # 1. Recalcul des probabilités de défaut XYZ avec le nouveau spread
    new_hr_xyz = (zc_xyz + spread_xyz_adj) / (1 - R)
    new_sp_xyz = np.exp(-new_hr_xyz * maturities)
    # Calcul des probabilités marginales (DP) : différence entre survie T-1 et T
    new_dp_xyz = np.diff(new_sp_xyz, prepend=1) * -1
    
    # 2. Recalcul de l'EPE avec la nouvelle volatilité
    new_epe = []
    for t in maturities:
        if t == 0:
            new_epe.append(0)
        else:
            dv01_res = dv01_total * (n_years - t) / n_years
            # Formule de Black avec ajustement de vol
            price = black_formula(strike, strike, vol_swaption + vol_adj, t)
            new_epe.append(price * 10000 * dv01_res)
    
    # 3. Calcul de la CVA (en utilisant les B0_Ti_Real extraits à la Question 4)
    # On multiplie chaque composante terme à terme
    cva_components = (1 - R) * df_q['B0_Ti_Real'] * new_epe * new_dp_xyz * df_q['SP_Bank']
    return cva_components.sum()

# --- Calcul des impacts ---

# A. Sensibilité à la Volatilité (+1%)
cva_vol_plus = compute_cva_sensi(vol_adj=0.01)
sensi_vol = cva_vol_plus - cva_total

# B. Sensibilité au Crédit de la contrepartie (+10 bps = 0.0010)
cva_credit_plus = compute_cva_sensi(spread_xyz_adj=0.0010)
sensi_credit = cva_credit_plus - cva_total

print(f"--- RÉSULTATS QUESTION 5 ---")
print(f"CVA de référence : {cva_total:,.2f} €")
print(f"Sensibilité Vega (Vol +1%) : {sensi_vol:,.2f} €")
print(f"Sensibilité Crédit (Spread +10bps) : {sensi_credit:,.2f} €")

--- RÉSULTATS QUESTION 5 ---
CVA de référence : 140,068.43 €
Sensibilité Vega (Vol +1%) : 5,394.52 €
Sensibilité Crédit (Spread +10bps) : 5,453.91 €


1. **Analyse de la Sensibilité Vega (+5 394,52 €) :** Cette valeur positive confirme que la CVA est "Long Vol".L'explication : Comme l'exposition ($EPE$) est calculée avec la formule de Black (comme une option), une hausse de la volatilité de 1 % augmente la probabilité que les taux s'écartent du strike.L'impact : Cela élargit l'enveloppe des gains potentiels pour la banque, et donc le risque de perte en cas de défaut. Ton résultat montre qu'une simple hausse de 1 % de la volatilité coûte à la banque plus de 5 300 € de provision supplémentaire.

2. **Analyse de la Sensibilité Crédit (+5 453,91 €) :** Cette valeur montre l'impact d'une dégradation de la santé financière de la contrepartie XYZ.L'explication : Un ajout de 10 bps sur le spread de crédit augmente mathématiquement la probabilité de défaut ($DP$).L'impact : Le risque de crédit pur pèse ici presque autant que le risque de volatilité (environ 5 400 €). C'est le "coût de la signature" : si XYZ devient plus risquée sur le marché, le contrat de la banque perd immédiatement de la valeur.

L'analyse des sensibilités démontre que la CVA de 140 068,43 € est un risque hybride. Elle est sensible à la fois aux conditions de marché (Vega) et à la qualité de crédit de la contrepartie (Delta Credit). On observe qu'une hausse de la volatilité des taux de 1 % a un impact financier quasi équivalent à une dégradation du spread de crédit de 10 bps. Cela justifie l'importance pour une banque de piloter dynamiquement ses limites de crédit, car une crise de volatilité sur les marchés augmente mécaniquement le risque de contrepartie, même sans dégradation intrinsèque du client.